In [1]:
!where python

d:\dev\github\mission16\.venv\Scripts\python.exe
C:\Users\kmw\AppData\Local\Programs\Python\Python310\python.exe
C:\Users\kmw\AppData\Local\Microsoft\WindowsApps\python.exe


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torch.quantization
import os
import copy
import torch.quantization as tq

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
# ResNet18(pretrained=True) 로드
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)  # CIFAR-10 (10 클래스)
model = model.to(device).eval()
print("ResNet18(pretrained=True) 로드 완료")

ResNet18(pretrained=True) 로드 완료


d:\dev\github\mission16\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\dev\github\mission16\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
# outputs 폴더 생성
os.makedirs('outputs/models', exist_ok=True)
print("outputs/models 폴더 생성 완료")

outputs/models 폴더 생성 완료


In [5]:
# FP32 모델 저장
try:
    torch.save(model.state_dict(), 'outputs/models/resnet18.pth')
    model_size = os.path.getsize('outputs/models/resnet18.pth') / (1024*1024)
    print(f"[성공] FP32 모델 저장: outputs/models/resnet18.pth ({model_size:.2f}MB)")
except Exception as e:
    print(f"[실패] FP32 모델 저장 오류: {e}")

[성공] FP32 모델 저장: outputs/models/resnet18.pth (42.73MB)


In [6]:
# 양자화 모델 저장 (Dynamic Quantization)
try:
    model_fp32_cpu = copy.deepcopy(model).cpu().eval()

    model_quant = torch.quantization.quantize_dynamic(
        model_fp32_cpu,
        {torch.nn.Linear},
        dtype=torch.qint8
    )

    torch.save(model_quant.state_dict(), 'outputs/models/resnet18_quant.pth')
    quant_size = os.path.getsize('outputs/models/resnet18_quant.pth') / (1024 * 1024)

    print(f"[성공] 동적 양자화 모델 저장: outputs/models/resnet18_quant.pth ({quant_size:.2f}MB)")

except Exception as e:
    print(f"[실패] 동적 양자화 모델 저장 오류: {e}")

[성공] 동적 양자화 모델 저장: outputs/models/resnet18_quant.pth (42.72MB)


C:\Users\kmw\AppData\Local\Temp\ipykernel_2344\2684578246.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quant = torch.quantization.quantize_dynamic(


In [7]:
# ONNX 변환
try:
    model_onnx = model.cpu().eval()
    dummy_input = torch.randn(1, 3, 224, 224)

    torch.onnx.export(
        model_onnx,
        dummy_input,
        'outputs/models/resnet18.onnx',
        input_names=['input'],
        output_names=['output'],
        opset_version=18,
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        },
        verbose=False
    )

    onnx_size = os.path.getsize('outputs/models/resnet18.onnx') / (1024 * 1024)
    print(f"[성공] ONNX 모델 저장: outputs/models/resnet18.onnx ({onnx_size:.2f}MB)")

except Exception as e:
    print(f"[실패] ONNX 변환 오류: {e}")


C:\Users\kmw\AppData\Local\Temp\ipykernel_2344\991139860.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0312 11:55:06.488000 2344 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0312 11:55:06.493000 2344 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0312 11:55:06.493000 2344 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'inpu

Applied 41 of general pattern rewrite rules.
[성공] ONNX 모델 저장: outputs/models/resnet18.onnx (0.09MB)


In [9]:
# 최종 확인
print("\n 생성된 파일 목록:")
for f in os.listdir('outputs/models'):
    size = os.path.getsize(f'outputs/models/{f}') / (1024*1024)
    print(f"  - {f}: {size:.2f}MB")

print("\n Basic Modeling 완료")


 생성된 파일 목록:
  - resnet18.onnx: 0.09MB
  - resnet18.onnx.data: 42.69MB
  - resnet18.pth: 42.73MB
  - resnet18_quant.pth: 42.72MB

 Basic Modeling 완료
